# Notebook 01 — Data Audit
**Thesis:** Fall Detection Generalization Across Vulnerable Populations  
**Purpose:** Inspect every raw dataset, confirm labels, count samples per subject/class/age-group, flag quality issues.  
Answers the feasibility notes in Delivery 1 §3.


## 0. Environment setup (Colab)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Adjust this path to where you stored the repo on your Drive
    REPO_ROOT = '/content/drive/MyDrive/fall-detection-generalization'
    DATA_ROOT = REPO_ROOT + '/data/raw'
else:
    import pathlib
    REPO_ROOT = str(pathlib.Path('..').resolve())
    DATA_ROOT = REPO_ROOT + '/data/raw'

print('Repo root:', REPO_ROOT)
print('Data root:', DATA_ROOT)

In [ ]:
if IN_COLAB:
    %pip install -q numpy pandas scipy matplotlib seaborn tqdm
    import sys
    sys.path.insert(0, REPO_ROOT)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm.notebook import tqdm

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

## 1. SisFall audit

In [ ]:
import sys, os
sys.path.insert(0, REPO_ROOT)
from src.data.sisfall import load_sisfall, trials_to_dataframe

SISFALL_ROOT = Path(DATA_ROOT) / 'sisfall'
print('SisFall path exists:', SISFALL_ROOT.exists())

In [ ]:
trials = load_sisfall(SISFALL_ROOT)
df_sis = trials_to_dataframe(trials)
print(f'Loaded {len(df_sis)} trials')
df_sis.head()

In [ ]:
print('=== Trial counts by age group and label ===')
print(df_sis.groupby(['age_group', 'is_fall']).size().rename('count').to_string())

In [ ]:
print('=== Subjects ===')
print(df_sis.groupby(['age_group', 'subject_id', 'is_fall']).size()
      .rename('trials').reset_index()
      .to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Trial counts
ct = df_sis.groupby(['age_group', 'is_fall']).size().unstack(fill_value=0)
ct.columns = ['ADL', 'Fall']
ct.plot(kind='bar', ax=axes[0], rot=0)
axes[0].set_title('SisFall: trial counts by age group')
axes[0].set_xlabel('')

# Signal durations
df_sis.boxplot(column='duration_s', by=['age_group', 'is_fall'], ax=axes[1], rot=20)
axes[1].set_title('Trial duration (s)')
axes[1].set_xlabel('')
plt.suptitle('')
plt.tight_layout()
plt.savefig(Path(REPO_ROOT) / 'results' / 'audit_sisfall_counts.png', bbox_inches='tight')
plt.show()

In [ ]:
print('=== Activity breakdown ===')
print(df_sis.groupby(['age_group', 'activity']).size().rename('n')
      .reset_index().sort_values(['age_group', 'activity']).to_string(index=False))

### 1.1 Sample signal plots
Plot one young fall, one elderly ADL and check for artefacts.

In [ ]:
from src.data.sisfall import SAMPLE_RATE

young_fall  = next(t for t in trials if t.age_group == 'young'  and t.is_fall)
elderly_adl = next(t for t in trials if t.age_group == 'elderly' and not t.is_fall)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
for ax, trial, title in [
    (axes[0], young_fall,  f'Young fall — {young_fall.activity} {young_fall.subject_id}'),
    (axes[1], elderly_adl, f'Elderly ADL — {elderly_adl.activity} {elderly_adl.subject_id}'),
]:
    t = np.arange(len(trial.signal)) / SAMPLE_RATE
    for i, lbl in enumerate(['ax (g)', 'ay (g)', 'az (g)']):
        ax.plot(t, trial.signal[:, i], label=lbl, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig(Path(REPO_ROOT) / 'results' / 'audit_sample_signals.png', bbox_inches='tight')
plt.show()

## 2. HAR70+ audit

In [ ]:
from src.data.har70plus import load_har70plus

HAR_ROOT = Path(DATA_ROOT) / 'har70plus'
print('HAR70+ path exists:', HAR_ROOT.exists())

if HAR_ROOT.exists():
    df_har = load_har70plus(HAR_ROOT)
    print(f'Loaded {len(df_har):,} rows from HAR70+')
    print(df_har.head())
    print('\nActivity counts:')
    print(df_har['activity_label'].value_counts().to_string())
else:
    print('HAR70+ not yet downloaded — skipping. See data/README.md.')

## 3. Laavanya dataset audit

In [ ]:
LAV_ROOT = Path(DATA_ROOT) / 'laavanya'
print('Laavanya path exists:', LAV_ROOT.exists())

if LAV_ROOT.exists():
    csvs = list(LAV_ROOT.glob('**/*.csv'))
    print(f'Found {len(csvs)} CSV files')
    for c in csvs[:3]:
        df_tmp = pd.read_csv(c, nrows=5)
        print(f'\n{c.name}  shape={df_tmp.shape}  columns={list(df_tmp.columns)}')
        print(df_tmp)
else:
    print('Laavanya not yet downloaded — skipping. See data/README.md.')

## 4. Audit summary

Fill in after running the cells above.

| Dataset | Subjects | Falls | ADLs | Sensor Hz | Notes |
|---|---|---|---|---|---|
| SisFall young | | | | 200 | |
| SisFall elderly | | | | 200 | |
| HAR70+ | | 0 | | 50 | ADL only |
| Laavanya | TBD | TBD | TBD | TBD | Needs manual check |

In [ ]:
# Save audit metadata for downstream notebooks
out_dir = Path(REPO_ROOT) / 'results'
out_dir.mkdir(exist_ok=True)
df_sis.to_csv(out_dir / 'sisfall_metadata.csv', index=False)
print('Metadata saved to results/sisfall_metadata.csv')